# SplitSnap 2: read a bill, check it, split it

Loads the fine-tuned model, reads one bill photo with per-field confidences, shows what the app would highlight for review, then splits the bill among named people with exact amounts and written explanations. Runs on a laptop, Colab or Kaggle with a GPU (CPU works, slowly).

You need the fine-tuned checkpoint folder (the README says where to put it). Set `CKPT` below, or the `CKPT` environment variable. On Colab, mount Drive or upload the folder first.

[Open in Colab](https://colab.research.google.com/github/atrijopal/naala_ml_tharki_ap/blob/main/notebooks/02_inference_and_split_demo.ipynb)

In [ ]:
import os, sys, json, pathlib, subprocess

REPO = "https://github.com/atrijopal/naala_ml_tharki_ap.git"
if "google.colab" in sys.modules or os.path.exists("/kaggle"):                     # hosted notebook: fetch the code once
    if not os.path.exists("proj"):
        subprocess.run(f"git clone -q {REPO} proj && pip install -q 'transformers>=4.44,<5' sentencepiece zss nltk scikit-learn", shell=True, check=True)
    os.chdir("proj")
root = pathlib.Path.cwd()                                                          # local: walk up to the repository root
while not (root / "splitsnap").exists() and root != root.parent:
    root = root.parent
os.chdir(root); sys.path.insert(0, str(root))

CKPT = os.environ.get("CKPT", "ckpt/best_fp16")        # folder with config.json, model.safetensors, tokenizer files
IMAGE = "examples/demo_bills/bill_07.jpg"              # any bill photo; the ten demo bills are generated, not real photographs
print("repository:", root, "| checkpoint exists:", os.path.exists(CKPT))

## 1. Load the model

In [ ]:
import torch
from splitsnap.model import load_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model, processor = load_model(CKPT)
model.to(device).eval()
if device == "cuda":
    model.half()
print("device:", device, "| parameters:", round(sum(p.numel() for p in model.parameters()) / 1e6), "M")

## 2. The photo and its quality check

The app asks for a retake when a photo is too small, blurred, dark, washed out or faint; the check is cheap and runs before the model.

In [ ]:
from PIL import Image
from splitsnap.preprocess import quality_report

img = Image.open(IMAGE).convert("RGB")
try:
    from IPython.display import display
    display(img.resize((img.width * 600 // img.height, 600)))
except Exception:
    pass
print(quality_report(img))

## 3. Read the bill, with a confidence for every field

In [ ]:
import time
from splitsnap.infer import generate_json

t = time.perf_counter()
with torch.inference_mode():
    bill, fields = generate_json(model, processor, img, device, 512, with_conf=True)
print(f"read in {time.perf_counter() - t:.2f} s on {device}\n")
print(json.dumps(bill, indent=2))

## 4. What the correction screen would highlight

A field's confidence is the lowest probability among the tokens that wrote it. The app highlights fields below the calibrated threshold (the lowest-scoring 8% on validation bills). The arithmetic check marks lines that do not add up and, for a misread rate, offers a one-tap fix.

In [ ]:
from splitsnap.validate import check_arithmetic

threshold = json.load(open("results/calibration.json"))["aggregators"]["p_min"]["flag_below"]
low = [(f["path"], f["value"], round(f["p_min"], 4)) for f in fields if f["p_min"] < threshold]
print(f"threshold {threshold}: {len(low)} of {len(fields)} fields highlighted")
for path, value, p in low:
    print(f"  {path:28s} {value!s:>12}   confidence {p}")
issues = check_arithmetic(bill)
print("\narithmetic issues:", issues if issues else "none: the bill adds up")

## 5. Split the bill

Amounts are integer paise. Items are divided by share units (a dish shared three ways gives each person one unit); every tax, service charge, discount and rounding line is divided in proportion to each person's pre-tax subtotal, with largest-remainder rounding so the parts add up exactly.

In [ ]:
from splitsnap.schema import ensure_schema
from splitsnap.split_engine import split_bill
from splitsnap.explain import charge_summary, explain_all

bill = ensure_schema(bill)
people = ["Riya", "Aman", "Sara"]
n = len(bill["items"])
assignments = {0: {p: 1 for p in people}}                    # first item shared three ways
if n > 1: assignments[1] = {"Riya": 2, "Aman": 1}            # second: two portions to Riya, one to Aman
for i in range(2, n - 1): assignments[i] = {people[i % 3]: 1}
if n > 2: assignments[n - 1] = {"Aman": 1, "Sara": 1}        # last shared by two

result = split_bill(bill, people, assignments)
for p in people:
    r = result["people"][p]
    print(f"{p:6s} items {r['pre_tax'] / 100:9.2f}   charges {r['charge_total'] / 100:8.2f}   total {r['final'] / 100:9.2f}")
total = sum(result["people"][p]["final"] for p in people)
print(f"\nsum of the amounts: {total / 100:.2f}   (bill total as read: {bill['grand_total']})")
print("\n" + "\n\n".join(explain_all(bill, result)))

## 6. Manual override

Any person's final amount can be overridden. The result is flagged as a manual adjustment, and the difference can be spread over the others.

In [ ]:
from splitsnap.split_engine import apply_overrides, to_paise

adjusted = apply_overrides(result, {"Riya": to_paise(300)}, rebalance=True)
for p in people:
    r = adjusted["people"][p]
    print(f"{p:6s} {r['final'] / 100:9.2f}   {'(adjusted by hand)' if r.get('manual') else ''}")
print("total:", sum(adjusted['people'][p]['final'] for p in people) / 100)

## 7. Optional: the same photo before fine-tuning

Downloads `donut-base` (about 800 MB) with our tokens and input size added but not fine-tuned. It has never seen our output format, so it writes the start token and then stops. Set `RUN_BASELINE = True` to try.

In [ ]:
RUN_BASELINE = False
if RUN_BASELINE:
    from splitsnap.model import build_model
    base, base_processor = build_model(1280, 960, 512)
    base.to(device).eval()
    if device == "cuda":
        base.half()
    from splitsnap.infer import generate_raw
    with torch.inference_mode():
        print(generate_raw(base, base_processor, img, device, 512))